# File Processing

In [1]:
from chamd import ChatReader
import pandas as pd
import os
import numpy as np
import random
import json
import torch
import csv
from sklearn.model_selection import train_test_split

In [2]:
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [3]:
RAW_DATA_FOLDER = 'datasetsRaw'
RAW_OUTPUT_FOLDER = 'datasetsPrep'
MEDICAL_FOLDER = 'Medical Datasets'
OTHER_FOLDER = 'Other Datasets'

In [4]:
medical_datasets = {}
other_datasets = {}
proc_medical_datasets = {}
proc_other_datasets = {}

## Medical Datasets

In [5]:
# # ----- Augmented Clinical Notes -----
# augmented_clinical_notes_df = pd.read_json(f"./{RAW_DATA_FOLDER}/{MEDICAL_FOLDER}/Augmented Clinical Notes/mediNote.json")
# augmented_clinical_notes = augmented_clinical_notes_df['full_note'].values.tolist()
# medical_datasets['augmentedClinicalNotes'] = augmented_clinical_notes

In [6]:
# ----- Clinical Dialogue Summarizations -----
clinical_dialogue_summarization_df = pd.read_csv(f"./{RAW_DATA_FOLDER}/{MEDICAL_FOLDER}/Clinical Dialogue Summarizations/MTS-Dialog-Augmented-TrainingSet-3-FR-and-ES-3603-Pairs-final.csv")
clinical_dialogue_summarization = clinical_dialogue_summarization_df['section_text'].values.tolist()
medical_datasets['clinicalDialogueSummarizations'] = clinical_dialogue_summarization

In [7]:
# ----- DementiaAudio -----
def get_chat(reader, file):
    chat = reader.read_file(file)
    chat_text = ""
    for line in chat.lines:
        if 'PAR' in str(line.metadata['speaker']):
            chat_text += (str(line.text)) + " "
    return chat_text

reader = ChatReader()
chat_dict = {'notes': [], 'label': []}
for folder in os.listdir(f"./{RAW_DATA_FOLDER}/{MEDICAL_FOLDER}/DementiaAudio"):
    if '.' not in folder:
        for inner_folder in os.listdir(f"./{RAW_DATA_FOLDER}/{MEDICAL_FOLDER}/DementiaAudio/{folder}"):
            if '.' not in inner_folder:
                for file in os.listdir(f"./{RAW_DATA_FOLDER}/{MEDICAL_FOLDER}/DementiaAudio/{folder}/{inner_folder}"):
                    if '.cha' in file:
                        temp_file = f"./{RAW_DATA_FOLDER}/{MEDICAL_FOLDER}/DementiaAudio/{folder}/{inner_folder}/{file}"
                        chat_dict['notes'].append(get_chat(reader, temp_file))
                        chat_dict['label'].append(folder)

label_coding = [1 if label == 'Dementia' else 0 for label in chat_dict['label']]
chat_dict['label_coding'] = label_coding
dementia_audio_df = pd.DataFrame(chat_dict)
dementia_audio = dementia_audio_df['notes'].values.tolist()
medical_datasets['dementiaAudio'] = dementia_audio

In [8]:
# ----- Medical Abstracts -----
medical_abstracts_train = pd.read_csv(f"./{RAW_DATA_FOLDER}/{MEDICAL_FOLDER}/Medical Abstracts/medical_tc_train.csv")
medical_abstracts_test = pd.read_csv(f"./{RAW_DATA_FOLDER}/{MEDICAL_FOLDER}/Medical Abstracts/medical_tc_test.csv")
medical_abstracts_df = pd.concat([medical_abstracts_train, medical_abstracts_test])
medical_abstracts = medical_abstracts_df['medical_abstract'].values.tolist()
medical_datasets['medicalAbstracts'] = medical_abstracts

In [9]:
# ----- SimSUM -----
sim_sum_df = pd.read_csv(f"./{RAW_DATA_FOLDER}/{MEDICAL_FOLDER}/SimSUM/SimSUM.csv", sep=";")
sim_sum = sim_sum_df['advanced_text'].values.tolist()
medical_datasets['simSUM'] = sim_sum

In [10]:
# ----- Synthetic Care Home Nurse Notes -----
synthetic_care_home_nurse_notes = []
for file in os.listdir(f"./{RAW_DATA_FOLDER}/{MEDICAL_FOLDER}/Synthetic Care Home Nurse Notes"):
    synthetic_care_home_nurse_notes.extend(pd.read_csv(f"./{RAW_DATA_FOLDER}/{MEDICAL_FOLDER}/Synthetic Care Home Nurse Notes/{file}")['report'].values.tolist())
medical_datasets['syntheticCareHomeNurseNotes'] = synthetic_care_home_nurse_notes 

## Other Datasets

In [11]:
# # ----- 20 NewsGroups -----
# newsgroups_df = pd.read_json(f"./{RAW_DATA_FOLDER}/{OTHER_FOLDER}/20 NewsGroups/20NewsGroups.json")
# newsgroups = newsgroups_df['text'].values.tolist()
# other_datasets['20NewsGroups'] = newsgroups

In [12]:
# # ----- BBC News -----
# bbc_df = pd.read_csv(f"./{RAW_DATA_FOLDER}/{OTHER_FOLDER}/BBC News/bbc-news-data.csv", sep='\t')
# bbc = bbc_df['content'].values.tolist()
# other_datasets['bbcNews'] = bbc

In [13]:
# # ----- Trump Tweets -----
# trump_df = pd.read_csv(f"./{RAW_DATA_FOLDER}/{OTHER_FOLDER}/Trump Tweets/tweets_01-08-2021.csv")
# trump = trump_df['text'].values.tolist()
# other_datasets['trumpTweets'] = trump

In [14]:
# ----- HuffPost News -----
huffpost_df = pd.read_json(f"./{RAW_DATA_FOLDER}/{OTHER_FOLDER}/HuffPost News/News_Category_Dataset_v3.json", lines=True)
huffpost = huffpost_df['short_description'].values.tolist()
other_datasets['huffPostNews'] = huffpost

In [15]:
# ----- Banking77 -----
banking_df = pd.read_csv(f"./{RAW_DATA_FOLDER}/{OTHER_FOLDER}/Banking77/banking77.csv")
banking = banking_df['text'].values.tolist()
other_datasets['banking77'] = banking

In [16]:
# ----- Atis -----
atis_df = pd.read_csv(f"./{RAW_DATA_FOLDER}/{OTHER_FOLDER}/Atis/atis_intents.csv", header=None, names=['category', 'text'])
atis = atis_df['text'].values.tolist()
other_datasets['atis'] = atis

In [17]:
# ----- Yahoo -----
yahoo_df = pd.read_json(f"./{RAW_DATA_FOLDER}/{OTHER_FOLDER}/Yahoo/nfL6.json")
yahoo = yahoo_df['answer'].values.tolist()
other_datasets['yahoo'] = yahoo

In [18]:
# ----- Clinc150 -----
path = f"./{RAW_DATA_FOLDER}/{OTHER_FOLDER}/Clinc150/data_full.json"
with open(path) as f:
    data = json.load(f)

clinc150 = []
# both regular labelled and out-of-scope examples are included
for key, values in data.items():
    for value in values:
        clinc150.append(value[0])

other_datasets['clinc150'] = clinc150

In [19]:
def make_dfs(unprocessed, output_dir, output_type, output_file_name,  val_size=0.15, test_size=0.15, random_state=RANDOM_STATE):
    # Format output_type for folder name.
    output_type = output_type.lower().split()
    output_type = f"{output_type[0]}{output_type[1][0].upper()}{output_type[1][1:]}"
    
    # Create directory.
    os.makedirs(f"./{output_dir}/{output_type}", exist_ok=True)
    
    # Convert to DataFrame.
    df = pd.DataFrame({'text': unprocessed})

    # First split off the test set
    train_val_df, test_df = train_test_split(df, test_size=test_size, random_state=random_state, shuffle=True)
    
    # Then split train/val
    val_relative_size = val_size / (1 - test_size)  # Adjust validation fraction relative to train+val
    train_df, val_df = train_test_split(train_val_df, test_size=val_relative_size, random_state=random_state, shuffle=True)

    # Dictionary to iterate over splits
    splits = {
        'train': train_df,
        'val': val_df,
        'test': test_df
    }

    # Save each split as CSV and JSON
    for split_name, split_df in splits.items():
        print(f"{split_name}: {len(split_df)}")
        csv_path = f"{output_dir}/{output_type}/{output_file_name}_{split_name}.csv"
        json_path = f"{output_dir}/{output_type}/{output_file_name}_{split_name}.json"

        split_df.to_csv(csv_path, index=False, lineterminator="\n", quoting=csv.QUOTE_ALL)
        split_df.to_json(json_path, orient="records", indent=2)

    print(f"Saved train/val/test splits to {output_dir}/{output_type}/")
    return len(train_df), len(test_df), len(val_df)

In [20]:
train_nums, test_nums, val_nums = [], [], []
for dataset_name, dataset in medical_datasets.items():
    train_num, test_num, val_num = make_dfs(dataset, RAW_OUTPUT_FOLDER, MEDICAL_FOLDER, dataset_name)
    train_nums.append(train_num)
    test_nums.append(test_num)
    val_nums.append(val_num)

for dataset_name, dataset in other_datasets.items():
    train_num, test_num, val_num = make_dfs(dataset, RAW_OUTPUT_FOLDER, OTHER_FOLDER, dataset_name)
    train_nums.append(train_num)
    test_nums.append(test_num)
    val_nums.append(val_num)

train: 2521
val: 541
test: 541
Saved train/val/test splits to datasetsPrep/medicalDatasets/
train: 383
val: 83
test: 83
Saved train/val/test splits to datasetsPrep/medicalDatasets/
train: 10106
val: 2166
test: 2166
Saved train/val/test splits to datasetsPrep/medicalDatasets/
train: 7000
val: 1500
test: 1500
Saved train/val/test splits to datasetsPrep/medicalDatasets/
train: 4047
val: 868
test: 868
Saved train/val/test splits to datasetsPrep/medicalDatasets/
train: 146668
val: 31429
test: 31430
Saved train/val/test splits to datasetsPrep/otherDatasets/
train: 9147
val: 1961
test: 1961
Saved train/val/test splits to datasetsPrep/otherDatasets/
train: 3484
val: 747
test: 747
Saved train/val/test splits to datasetsPrep/otherDatasets/
train: 61152
val: 13105
test: 13105
Saved train/val/test splits to datasetsPrep/otherDatasets/
train: 16589
val: 3556
test: 3555
Saved train/val/test splits to datasetsPrep/otherDatasets/


In [28]:
min(train_nums + test_nums + val_nums)

83